# Step-by-Step Audio Transcription & Translation

This notebook demonstrates how to transcribe audio in Russian, Arabic, or Chinese and translate it to English using Whisper and HuggingFace models. It also shows how to write both the original and translated text to a file using multithreading.

## Step 1: Install Required Packages
We need `faster-whisper`, `transformers`, `sentencepiece`, `srt`, and `tqdm`.

In [ ]:
!pip install faster-whisper transformers sentencepiece srt tqdm

## Step 2: Import Libraries and Define Helper Functions

In [ ]:
import threading
import srt
from tqdm import tqdm
from faster_whisper import WhisperModel
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from pathlib import Path
import concurrent.futures
import logging
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')

SUPPORTED = {
    "ar": "Helsinki-NLP/opus-mt-ar-en",
    "ru": "Helsinki-NLP/opus-mt-ru-en",
    "zh": "Helsinki-NLP/opus-mt-zh-en",
}

def load_mt(src_lang):
    model_name = SUPPORTED[src_lang]
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSeq2SeqLM.from_pretrained(model_name)
    return tok, mdl

def translate_chunks_parallel(chunks, tok, mdl, max_len=512):
    def translate_one(ch):
        inp = tok(ch, return_tensors="pt", truncation=True, max_length=max_len)
        gen = mdl.generate(**inp, max_new_tokens=512)
        return tok.batch_decode(gen, skip_special_tokens=True)[0]
    with concurrent.futures.ThreadPoolExecutor() as executor:
        return list(executor.map(translate_one, chunks))

def write_text(filepath, text, label):
    with open(filepath, 'a', encoding='utf-8') as f:
        f.write(f"{label}:\n{text}\n\n")

def write_combined_transcript_multithread(transcript_path, transcript_text, english_translation, lang_label):
    open(transcript_path, "w", encoding="utf-8").close()
    threads = [
        threading.Thread(target=write_text, args=(transcript_path, transcript_text, f"Original {lang_label}")),
        threading.Thread(target=write_text, args=(transcript_path, english_translation or "", "English Translation"))
    ]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

## Step 3: Transcribe Audio and Detect Language
Use Whisper to transcribe the audio file and detect its language.

In [ ]:
audio_path = "Russian.mp3"  # Change to your audio file
model = WhisperModel("medium", device="cpu", compute_type="float32")
segments, info = model.transcribe(audio_path, language=None, beam_size=5, vad_filter=True, word_timestamps=False)
segments_list = list(segments)
transcript_text = " ".join([seg.text.strip() for seg in segments_list])
detected_lang = info.language
print(f"Detected language: {detected_lang}")

## Step 4: Translate Transcript to English
Load the appropriate translation model and translate the transcript using parallel threads.

In [ ]:
src_mt = "zh" if detected_lang.startswith("zh") else detected_lang
if src_mt in SUPPORTED:
    tok, mdl = load_mt(src_mt)
    english_translation = translate_chunks_parallel([transcript_text], tok, mdl)[0]
else:
    english_translation = None
print("English Translation:\n", english_translation)

## Step 5: Write Both Transcripts to Output File Using Multithreading

In [ ]:
output_path = "nlp_out/Russian.ru.txt"  # Change as needed
lang_label = detected_lang if detected_lang not in ["ar", "ru", "zh"] else {"ar": "Arabic", "ru": "Russian", "zh": "Chinese"}[detected_lang]
write_combined_transcript_multithread(output_path, transcript_text, english_translation, lang_label)
print(f"Written to {output_path}")

## Step 6: (Optional) View Output
You can now open the output file and see both the original and translated transcript.